# Window-Size Ablation — WINDOW_SIZE=50 (base VAE, no scoring tricks)

**Why this is the last legitimate base-VAE attempt**: every prior experiment
(multi-seed, extended features, max-pooling, two hyperparameter searches,
reconstruction-probability) held `WINDOW_SIZE=30` fixed and only varied the
model. This is the one dimension the proposal explicitly permits (30-60
timesteps) that was never touched. A different window size changes what
information is actually *in* each sample — a genuinely different axis, not
another model variant.

**Isolating the variable properly**: `hidden1=64`, `hidden2=32`,
`latent_dim=32`, `beta_max=0.01` are all held at the SAME values as the
deployed model. Only `WINDOW_SIZE` changes (30 -> 50). This means re-running
windowing + PCA from scratch (a 50-step window flattens to 350 raw dims,
not 210, so PCA has to be refit), then training a fresh VAE on that new
representation with the *same* hyperparameters as the deployed model — not a
fresh hyperparameter search, which would confound the window-size effect
with yet another tuning variable.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
import pickle, joblib, os, time
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                              recall_score, f1_score, precision_recall_curve)

torch.manual_seed(42)
np.random.seed(42)

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
MODEL_DIR = os.path.join(BASE, 'models')
OUT_DIR   = os.path.join(BASE, 'experiments')

DEVICE = torch.device('cpu')
NEW_WINDOW_SIZE = 50   # up from 30 - the proposal's upper bound
STRIDE = 1
PCA_VARIANCE = 0.99

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache',
]
SPLIT_FILES = {
    'cc1_train': 'cc1_train.csv', 'cc1_val': 'cc1_val.csv', 'cc1_test': 'cc1_test.csv',
    'drift_cc2': 'drift_complex_case2.csv',
}
ALL_SETS = ['cc1_test', 'drift_cc2']

HIDDEN1, HIDDEN2, LATENT_DIM, BETA_MAX = 64, 32, 32, 0.01   # SAME as the deployed model - only window size changes
WARMUP_EPOCHS, FULL_MAX_EPOCHS, FULL_PATIENCE = 10, 300, 20
LR, BATCH_SIZE, CLIP = 1e-3, 512, 20.0

print(f'NEW_WINDOW_SIZE={NEW_WINDOW_SIZE}  (deployed model uses 30)')
print(f'Fixed hyperparameters (unchanged from deployed): hidden1={HIDDEN1}, hidden2={HIDDEN2}, latent_dim={LATENT_DIM}, beta_max={BETA_MAX}')

NEW_WINDOW_SIZE=50  (deployed model uses 30)
Fixed hyperparameters (unchanged from deployed): hidden1=64, hidden2=32, latent_dim=32, beta_max=0.01


## Step 1 — Rebuild windows at size 60 (gap-aware, same logic as `windowing_pca.ipynb`)

In [2]:
def build_windows(container_df, feature_cols, window_size, stride):
    data_arr = container_df[feature_cols].values.astype(np.float32)
    labels = container_df['label'].values
    ftypes = container_df['failure_type'].values.astype(object)
    is_gap = container_df['is_gap'].values
    n = len(data_arr)
    X, y, ft = [], [], []
    for i in range(0, n - window_size + 1, stride):
        if is_gap[i:i+window_size].any():
            continue
        X.append(data_arr[i:i+window_size])
        window_labels = labels[i:i+window_size]
        y.append(int(window_labels.any()))
        w_types = sorted({t for t in ftypes[i:i+window_size] if isinstance(t, str)})
        ft.append(','.join(w_types) if w_types else None)
    if not X:
        return (np.empty((0, window_size, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.int64), np.array([], dtype=object))
    return np.stack(X), np.array(y, dtype=np.int64), np.array(ft, dtype=object)

def window_split(df, feature_cols, window_size, stride):
    Xs, ys, fts = [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        X, y, ft = build_windows(g, feature_cols, window_size, stride)
        if len(X):
            Xs.append(X); ys.append(y); fts.append(ft)
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(fts)

windowed = {}
for name, fname in SPLIT_FILES.items():
    d = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    X, y, ft = window_split(d, FEATURE_COLS, NEW_WINDOW_SIZE, STRIDE)
    windowed[name] = {'X': X, 'y': y, 'ft': ft}
    n_anom = int(y.sum())
    print(f'  {name:10s}: {len(d):>7,} rows -> {len(y):>7,} windows  ({n_anom:,} anomaly windows, {n_anom/max(len(y),1)*100:.2f}%)')

for name in windowed:
    X = windowed[name]['X']
    windowed[name]['X_flat'] = X.reshape(len(X), -1)
    print(f'  {name:10s}: {X.shape} -> {windowed[name]["X_flat"].shape}')

  cc1_train : 156,479 rows -> 152,996 windows  (0 anomaly windows, 0.00%)
  cc1_val   :  22,356 rows ->  21,033 windows  (0 anomaly windows, 0.00%)
  cc1_test  :  44,968 rows ->  43,645 windows  (256 anomaly windows, 0.59%)
  drift_cc2 :  77,760 rows ->  76,437 windows  (960 anomaly windows, 1.26%)
  cc1_train : (152996, 50, 7) -> (152996, 350)
  cc1_val   : (21033, 50, 7) -> (21033, 350)
  cc1_test  : (43645, 50, 7) -> (43645, 350)
  drift_cc2 : (76437, 50, 7) -> (76437, 350)


## Step 2 — Fit PCA fresh (99% variance + whiten, fit on `cc1_train` only)

The flattened dimension is now 60x7=420 (up from 210), so this cannot reuse
`cc1_pca.pkl` — a fresh fit is required, following the exact same
methodology as `windowing_pca.ipynb`.

In [3]:
X_train_flat = windowed['cc1_train']['X_flat']
print(f'Fitting PCA on {len(X_train_flat):,} cc1_train windows (all normal) ...')
print(f'Input dimension: {X_train_flat.shape[1]} (= {NEW_WINDOW_SIZE} timesteps x {len(FEATURE_COLS)} features)')

pca = PCA(n_components=PCA_VARIANCE, svd_solver='full', whiten=True, random_state=42)
pca.fit(X_train_flat)
n_components = pca.n_components_
print(f'Components retained for {PCA_VARIANCE*100:.0f}% variance: {n_components}  '
      f'(compression {X_train_flat.shape[1]} -> {n_components})')
print(f'(Deployed 30-step model: 210 -> 26 components, for comparison)')

for name in windowed:
    windowed[name]['X_pca'] = pca.transform(windowed[name]['X_flat']).astype(np.float32)
    print(f'  {name:10s}: {windowed[name]["X_flat"].shape} -> {windowed[name]["X_pca"].shape}')

Fitting PCA on 152,996 cc1_train windows (all normal) ...
Input dimension: 350 (= 50 timesteps x 7 features)
Components retained for 99% variance: 40  (compression 350 -> 40)
(Deployed 30-step model: 210 -> 26 components, for comparison)
  cc1_train : (152996, 350) -> (152996, 40)
  cc1_val   : (21033, 350) -> (21033, 40)
  cc1_test  : (43645, 350) -> (43645, 40)
  drift_cc2 : (76437, 350) -> (76437, 40)


## Step 3 — VAE class + training (identical architecture logic to `train_vae.ipynb`, only `input_dim` differs)

In [4]:
INPUT_DIM = n_components

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

def vae_loss(recon, x, mu, logvar, beta):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon_loss + beta * kl, recon_loss, kl

data = {name: np.clip(windowed[name]['X_pca'], -CLIP, CLIP).astype(np.float32) for name in windowed}
X_train_t = torch.from_numpy(data['cc1_train'])
X_val_t   = torch.from_numpy(data['cc1_val'])
labels = {name: windowed[name]['y'] for name in windowed}
assert (labels['cc1_train'] == 0).all()
print(f'INPUT_DIM={INPUT_DIM}  X_train={X_train_t.shape}  X_val={X_val_t.shape}')

def train_vae(latent_dim, max_epochs, patience, beta_max, verbose=True):
    torch.manual_seed(42)
    model = VAE(INPUT_DIM, HIDDEN1, HIDDEN2, latent_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
    best_val, best_state, patience_ctr = float('inf'), None, 0

    for epoch in range(max_epochs):
        beta = min(1.0, (epoch + 1) / WARMUP_EPOCHS) * beta_max
        model.train()
        for (xb,) in loader:
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            loss, rloss, kl = vae_loss(recon, xb, mu, logvar, beta)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            recon, mu, logvar = model(X_val_t)
            vloss, vrecon, vkl = vae_loss(recon, X_val_t, mu, logvar, beta)
        if vloss.item() < best_val - 1e-6:
            best_val, best_state, patience_ctr = vloss.item(), {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                if verbose:
                    print(f'    early stop at epoch {epoch+1} (best val_loss={best_val:.4f})')
                break
        if verbose and (epoch % 20 == 0 or epoch == max_epochs - 1):
            print(f'    epoch {epoch+1:3d}  beta={beta:.4f}  train_kl={kl.item():.4f}  val={vloss.item():.4f}')
    model.load_state_dict(best_state)
    return model, best_val, epoch + 1

print('Training function defined.')

INPUT_DIM=40  X_train=torch.Size([152996, 40])  X_val=torch.Size([21033, 40])
Training function defined.


## Step 4 — Full training, SAME hyperparameters as the deployed model (latent_dim=32, beta_max=0.01)

In [5]:
print(f'Training VAE at WINDOW_SIZE={NEW_WINDOW_SIZE}, INPUT_DIM={INPUT_DIM}, latent_dim={LATENT_DIM}, beta_max={BETA_MAX} ...')
t0 = time.time()
model, best_val, n_epochs = train_vae(LATENT_DIM, FULL_MAX_EPOCHS, FULL_PATIENCE, BETA_MAX, verbose=True)
elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min.  epochs={n_epochs}  val_loss={best_val:.4f}')

with torch.no_grad():
    mse_train = model.anomaly_score(X_train_t).numpy()
    mse_val = model.anomaly_score(X_val_t).numpy()
mu_train, sigma_train = float(mse_train.mean()), float(mse_train.std())
print(f'mu_train={mu_train:.5f}  sigma_train={sigma_train:.5f}')

Training VAE at WINDOW_SIZE=50, INPUT_DIM=40, latent_dim=32, beta_max=0.01 ...
    epoch   1  beta=0.0010  train_kl=47.2370  val=1.1668
    epoch  21  beta=0.0100  train_kl=15.5147  val=0.5894
    epoch  41  beta=0.0100  train_kl=17.5447  val=0.4533
    epoch  61  beta=0.0100  train_kl=17.0472  val=0.4365
    epoch  81  beta=0.0100  train_kl=17.3146  val=0.4290
    epoch 101  beta=0.0100  train_kl=17.3734  val=0.4250
    epoch 121  beta=0.0100  train_kl=17.3020  val=0.4200
    epoch 141  beta=0.0100  train_kl=17.3264  val=0.4167
    epoch 161  beta=0.0100  train_kl=17.6746  val=0.4128
    epoch 181  beta=0.0100  train_kl=17.4833  val=0.4092
    epoch 201  beta=0.0100  train_kl=17.1737  val=0.4125
    epoch 221  beta=0.0100  train_kl=17.3361  val=0.4120
    epoch 241  beta=0.0100  train_kl=17.8067  val=0.4075
    early stop at epoch 260 (best val_loss=0.4066)

Done in 9.3 min.  epochs=260  val_loss=0.4066
mu_train=0.14942  sigma_train=0.13687


## Step 5 — Full evaluation, identical protocol to `vae_eval.ipynb` — honest comparison to the deployed (30-step) model

In [6]:
val_p99 = float(np.percentile(mse_val, 99))
print(f'val_p99 threshold: {val_p99:.5f}')

def evaluate(scores, y_true, threshold):
    pred = (scores > threshold).astype(int)
    p, r, _ = precision_recall_curve(y_true, scores)
    f1s = 2 * p * r / (p + r + 1e-12)
    oracle_f1 = float(f1s[np.argmax(f1s)])
    return {
        'auc_roc': roc_auc_score(y_true, scores), 'auc_pr': average_precision_score(y_true, scores),
        'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0), 'oracle_f1': oracle_f1,
    }

w50_results = {}
mse_by_set = {}
for name in ALL_SETS:
    X_t = torch.from_numpy(data[name])
    with torch.no_grad():
        mse = model.anomaly_score(X_t).numpy()
    mse_by_set[name] = mse
    w50_results[name] = evaluate(mse, labels[name], val_p99)

deployed_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))

print(f'\n{"set":12s} {"model":16s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s} {"OracleF1":>9s}')
for name in ALL_SETS:
    t = w50_results[name]
    d_auc = deployed_eval['auc'][name]
    d_pr = deployed_eval['precision_recall'][name]['val_p99']
    d_oracle = deployed_eval['oracle_ceiling'][name]['f1']
    print(f'{name:12s} {"window=50":16s} {t["auc_pr"]:8.4f} {t["auc_roc"]:9.4f} {t["f1"]:7.3f} {t["precision"]:10.3f} {t["recall"]:8.3f} {t["oracle_f1"]:9.4f}')
    print(f'{name:12s} {"window=30 (deployed)":16s} {d_auc["auc_pr"]:8.4f} {d_auc["auc_roc"]:9.4f} {d_pr["f1"]:7.3f} {d_pr["precision"]:10.3f} {d_pr["recall"]:8.3f} {d_oracle:9.4f}')
    print(f'  -> PR-AUC change: {t["auc_pr"]-d_auc["auc_pr"]:+.4f}   F1 change: {t["f1"]-d_pr["f1"]:+.3f}   Oracle-F1 change: {t["oracle_f1"]-d_oracle:+.4f}\n')

val_p99 threshold: 0.90259

set          model              PR-AUC   ROC-AUC      F1  Precision   Recall  OracleF1
cc1_test     window=50          0.7342    0.9623   0.730      0.816    0.660    0.7571
cc1_test     window=30 (deployed)   0.6014    0.8763   0.618      0.630    0.605    0.6857
  -> PR-AUC change: +0.1328   F1 change: +0.112   Oracle-F1 change: +0.0714

drift_cc2    window=50          0.0956    0.8443   0.103      0.056    0.682    0.1620
drift_cc2    window=30 (deployed)   0.4089    0.8812   0.276      0.168    0.772    0.5147
  -> PR-AUC change: -0.3134   F1 change: -0.173   Oracle-F1 change: -0.3527



## Step 6 — Per-fault-type recall, window=50 vs. deployed (window=30)

In [7]:
deployed_fault = deployed_eval['per_fault_recall']

print(f'{"set":12s} {"fault_type":14s} {"n":>5s} {"deployed(w=30)":>15s} {"w=50":>8s}')
for name in ALL_SETS:
    pred = (mse_by_set[name] > val_p99).astype(int)
    ft = windowed[name]['ft']
    types_present = sorted({v for v in ft if isinstance(v, str)})
    for ftype in types_present:
        mask = ft == ftype
        n = int(mask.sum())
        rec_w50 = pred[mask].mean() if n > 0 else float('nan')
        rec_deployed = deployed_fault[name].get(ftype, {}).get('recall', float('nan'))
        print(f'{name:12s} {ftype:14s} {n:5d} {rec_deployed:15.3f} {rec_w50:8.3f}')
    print()

set          fault_type         n  deployed(w=30)     w=50
cc1_test     cpu               88           0.580    0.432
cc1_test     memory            76           0.763    0.789
cc1_test     pod-failure       92           0.500    0.772

drift_cc2    cpu              267           0.908    0.798
drift_cc2    memory           534           0.713    0.725
drift_cc2    pod-failure      159           0.737    0.346



## Step 7 — Save (does NOT overwrite the deployed 30-step model)

In [8]:
save_results = {
    'window_size': NEW_WINDOW_SIZE,
    'pca_n_components': n_components,
    'model_meta': {
        'input_dim': INPUT_DIM, 'hidden1': HIDDEN1, 'hidden2': HIDDEN2, 'latent_dim': LATENT_DIM,
        'beta_max': BETA_MAX, 'clip': CLIP, 'mu_train': mu_train, 'sigma_train': sigma_train,
        'val_p99': val_p99, 'epochs_trained': n_epochs,
    },
    'results': w50_results,
    'deployed_comparison': {
        name: {'auc': deployed_eval['auc'][name], 'precision_recall': deployed_eval['precision_recall'][name]['val_p99'],
               'oracle_f1': deployed_eval['oracle_ceiling'][name]['f1']}
        for name in ALL_SETS
    },
}
out_path = os.path.join(OUT_DIR, 'window_size_50_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

model_path = os.path.join(OUT_DIR, 'vae_cc1_window50.pt')
torch.save(model.state_dict(), model_path)
pca_path = os.path.join(OUT_DIR, 'cc1_pca_window50.pkl')
joblib.dump({'pca': pca, 'window_size': NEW_WINDOW_SIZE, 'n_components': n_components}, pca_path)
print(f'Model saved -> {model_path}  (NOT deployed - models/vae_cc1.pt is unchanged)')
print(f'PCA saved -> {pca_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\window_size_50_results.pkl
Model saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\vae_cc1_window50.pt  (NOT deployed - models/vae_cc1.pt is unchanged)
PCA saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\cc1_pca_window50.pkl


## How to read this

- **Only `WINDOW_SIZE` changed** (30 -> 50); architecture hyperparameters were
  held at the deployed model's exact values, so any difference in Step 5's
  results is attributable to the window-size change alone, not confounded
  with a fresh (and possibly misleading) hyperparameter search.
- **This is the last untested, proposal-permitted lever for the base VAE.**
  If this doesn't beat the deployed model either, that closes out the base-
  VAE-alone improvement question with strong, convergent evidence across
  seven independent, well-motivated attempts — a legitimate, reportable
  research conclusion in its own right.
- Nothing here overwrites `models/vae_cc1.pt` or `models/cc1_pca.pkl`.